In [ ]:
import json
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import pydeck as pdk
import pandas as pd
import numpy as np
from shapely.geometry import box

# Load master dataset
gdf = gpd.read_file("data/election_2026_master.geojson")

# Ensure STATEFP is a string zero-padded to 2 digits
#gdf["STATEFP"] = gdf["STATEFP"].astype(str).str.zfill(2)

# Retain only the 50 states + DC (FIPS codes <= 56)
US_STATE_FIPS = [str(i).zfill(2) for i in range(1, 57) if i not in [3, 7, 14, 43, 52]]
gdf = gdf[gdf["STATEFP"].isin(US_STATE_FIPS)].copy()

# Calculate provisional fallbacks across entire dataset
gdf["display_party"] = gdf["winner_party"].replace("", np.nan).fillna(gdf["leader_party"]).fillna("OTHER")

In [ ]:
# -------------------------------------------------------------
# CELL 1: Utility functions to reposition and rescale Alaska and Hawaii
# -------------------------------------------------------------
# Approach (same as the classic "Albers USA" composite):
#   1. Project the lower 48 with CONUS Albers (EPSG:5070).
#   2. Project Alaska with its OWN Albers (EPSG:3338) and Hawaii with its own
#      Albers (ESRI:102007), so each looks upright/undistorted instead of the
#      tilted, stretched shape it gets under a CONUS projection.
#   3. Build one inset frame in the empty area below CA/AZ: a rectangle whose
#      top-right corner is cut diagonally to follow the coast, with a Hawaii
#      cell in its top-left corner. Shared edges are drawn once, so no
#      borders overlap.
#   4. Scale + translate AK/HI into their parts of that frame.
#   5. Keep everything in projected metres for plotting. Do NOT convert back to
#      EPSG:4326 - the moved shapes are no longer at real lat/lon, and
#      reprojecting them warps and misplaces them.
from shapely.affinity import scale as shp_scale, translate as shp_translate
from shapely.geometry import LineString, Polygon
import warnings

L48_CRS = "EPSG:5070"    # NAD83 / Conus Albers
AK_CRS = "EPSG:3338"     # NAD83 / Alaska Albers (handles the Aleutians across the 180th meridian)
HI_CRS = "ESRI:102007"   # Hawaii Albers Equal Area Conic

# Inset layout, as fractions of the lower-48 bounding box
# (x: 0 = westernmost point, 1 = easternmost; y: 0 = southernmost, 1 = northernmost).
INSET_LAYOUT = dict(
    frame_x0=-0.01, frame_x1=0.375,   # left/right edges of the inset frame
    frame_y0=-0.135, frame_y1=0.24,   # bottom/top edges (negative = below the lower 48)
    cut_top_x=0.225,                  # where the diagonal leaves the top edge...
    cut_right_y=0.07,                # ...and where it meets the right edge
    hi_cell_x1=0.145,                 # right edge of the Hawaii cell
    hi_cell_y0=0.07,                 # bottom edge of the Hawaii cell
    cell_pad=0.08,                    # padding inside a cell, as a fraction of its size
    ak_x0=0.015, ak_y0=-0.12,        # lower-left corner of Alaska's bounding box
    ak_w=0.335,                       # width of Alaska's bounding box (incl. Aleutians)
)

def safe_margin(row):
    m = row.get("margin")
    if pd.notna(m) and m != 0:
        return m
    if row.get("display_party") == "DEM": return 0.8
    if row.get("display_party") == "REP": return -0.8
    return 0.0

def _inset_geometry(l48_bounds, L=INSET_LAYOUT):
    """Frame polygon, Hawaii cell polygon and the divider lines, in EPSG:5070 metres."""
    lx0, ly0, lx1, ly1 = l48_bounds
    lw, lh = lx1 - lx0, ly1 - ly0
    X = lambda f: lx0 + f * lw
    Y = lambda f: ly0 + f * lh
    frame = Polygon([
        (X(L["frame_x0"]), Y(L["frame_y0"])),
        (X(L["frame_x1"]), Y(L["frame_y0"])),
        (X(L["frame_x1"]), Y(L["cut_right_y"])),
        (X(L["cut_top_x"]), Y(L["frame_y1"])),
        (X(L["frame_x0"]), Y(L["frame_y1"])),
    ])
    hi_cell = box(X(L["frame_x0"]), Y(L["hi_cell_y0"]), X(L["hi_cell_x1"]), Y(L["frame_y1"]))
    dividers = [
        LineString([(X(L["hi_cell_x1"]), Y(L["frame_y1"])), (X(L["hi_cell_x1"]), Y(L["hi_cell_y0"]))]),
        LineString([(X(L["frame_x0"]), Y(L["hi_cell_y0"])), (X(L["hi_cell_x1"]), Y(L["hi_cell_y0"]))]),
    ]
    return frame, hi_cell, dividers

def _place(part, k, ref_xy, target_xy):
    """Uniformly scale `part` by k about ref_xy, then move ref_xy to target_xy."""
    part = part.copy()
    part.geometry = [
        shp_translate(shp_scale(g, k, k, origin=ref_xy),
                      xoff=target_xy[0] - ref_xy[0], yoff=target_xy[1] - ref_xy[1])
        for g in part.geometry
    ]
    return part.set_crs(L48_CRS, allow_override=True)

def get_shifted_us_gdf(gdf_input, drop_nw_hawaiian_islands=True, L=INSET_LAYOUT):
    """Return a composite (lower 48 + AK/HI insets) GeoDataFrame in EPSG:5070."""
    ak_mask = gdf_input["STATEFP"] == "02"
    hi_mask = gdf_input["STATEFP"] == "15"

    lower48 = gdf_input[~ak_mask & ~hi_mask].to_crs(L48_CRS)
    l48_bounds = lower48.total_bounds
    lx0, ly0, lx1, ly1 = l48_bounds
    lw, lh = lx1 - lx0, ly1 - ly0
    frame, hi_cell, _ = _inset_geometry(l48_bounds, L)
    parts = [lower48]

    if ak_mask.any():
        ak = gdf_input[ak_mask].to_crs(AK_CRS)
        ax0, ay0, ax1, ay1 = ak.total_bounds
        k = (L["ak_w"] * lw) / (ax1 - ax0)
        ak = _place(ak, k, (ax0, ay0), (lx0 + L["ak_x0"] * lw, ly0 + L["ak_y0"] * lh))
        parts.append(ak)

    if hi_mask.any():
        hi = gdf_input[hi_mask].copy()
        if drop_nw_hawaiian_islands:
            # HI-02 includes the uninhabited NW Hawaiian Islands, which run ~1,900 km
            # toward Midway. Keeping them shrinks the main islands to specks.
            hi = gpd.clip(hi, box(-161.0, 18.5, -154.5, 22.5))
        hi = hi.to_crs(HI_CRS)
        # Fit Hawaii inside its cell (with padding), centred.
        hx0, hy0, hx1, hy1 = hi.total_bounds
        cx0, cy0, cx1, cy1 = hi_cell.bounds
        pad = L["cell_pad"]
        k = min((1 - 2 * pad) * (cx1 - cx0) / (hx1 - hx0),
                (1 - 2 * pad) * (cy1 - cy0) / (hy1 - hy0))
        hi = _place(hi, k, ((hx0 + hx1) / 2, (hy0 + hy1) / 2), ((cx0 + cx1) / 2, (cy0 + cy1) / 2))
        parts.append(hi)

    out = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=L48_CRS)
    _check_layout(out, L)
    return out

def _check_layout(gdf_shifted, L=INSET_LAYOUT):
    """Warn if a layout tweak makes the frame touch the lower 48 or an inset spill out of its area."""
    is_ak = gdf_shifted["STATEFP"] == "02"
    is_hi = gdf_shifted["STATEFP"] == "15"
    l48 = gdf_shifted[~is_ak & ~is_hi]
    frame, hi_cell, _ = _inset_geometry(l48.total_bounds, L)
    if frame.intersects(l48.union_all()):
        warnings.warn("Inset frame overlaps the lower 48 - adjust INSET_LAYOUT (cut_* / frame_*).")
    ak_area = frame.difference(hi_cell)
    if is_ak.any() and not gdf_shifted[is_ak].union_all().within(ak_area):
        warnings.warn("Alaska spills outside its part of the frame - adjust ak_x0/ak_y0/ak_w.")
    if is_hi.any() and not gdf_shifted[is_hi].union_all().within(hi_cell):
        warnings.warn("Hawaii spills outside its cell - increase cell_pad or the cell size.")

def add_inset_boxes(ax, gdf_shifted, color="#555555", linewidth=0.9):
    """Draw the inset frame and the Hawaii cell divider as solid lines (each edge drawn once)."""
    l48 = gdf_shifted[~gdf_shifted["STATEFP"].isin(["02", "15"])]
    frame, _, dividers = _inset_geometry(l48.total_bounds)
    gpd.GeoSeries([frame.exterior] + dividers, crs=gdf_shifted.crs).plot(
        ax=ax, color=color, linewidth=linewidth, linestyle="-"
    )

def frame_axes(ax, gdf_shifted, pad_frac=0.01):
    """Fit the axes to the composite map + inset frame (replaces the hard-coded lat/lon limits)."""
    l48 = gdf_shifted[~gdf_shifted["STATEFP"].isin(["02", "15"])]
    frame, _, _ = _inset_geometry(l48.total_bounds)
    geoms = pd.concat([gdf_shifted.geometry, gpd.GeoSeries([frame], crs=gdf_shifted.crs)])
    x0, y0, x1, y1 = gpd.GeoSeries(geoms, crs=gdf_shifted.crs).total_bounds
    px, py = pad_frac * (x1 - x0), pad_frac * (y1 - y0)
    ax.set_xlim(x0 - px, x1 + px)
    ax.set_ylim(y0 - py, y1 + py)
    ax.set_aspect("equal")
    ax.axis("off")

# Set display margin
gdf["display_margin"] = gdf.apply(safe_margin, axis=1)

# Apply spatial shift to reposition AK & HI
gdf_shifted = get_shifted_us_gdf(gdf)

In [ ]:
# -------------------------------------------------------------
# CELL 2: Binary Blue-Red Color Map
# -------------------------------------------------------------
party_colors = {"DEM": "#2E5B88", "REP": "#D23B33", "OTHER": "#808080"}
gdf_shifted["color"] = gdf_shifted["display_party"].map(party_colors).fillna("#808080")

fig, ax = plt.subplots(1, 1, figsize=(15, 9))
gdf_shifted.plot(
    ax=ax,
    color=gdf_shifted["color"],
    edgecolor="black",
    linewidth=0.2
)
add_inset_boxes(ax, gdf_shifted)
frame_axes(ax, gdf_shifted)

ax.set_title("2026 House Seats: Binary Winner / Leading Party", fontsize=14)
plt.tight_layout()
plt.savefig("images/2026-Binary-Blue-Red-Color-Map.png", dpi=300, bbox_inches="tight", transparent=True, pad_inches=0.1)
plt.show()

In [ ]:
# -------------------------------------------------------------
# CELL 3: Blue-Red Gradient Map (Margin of Victory)
# -------------------------------------------------------------
custom_cmap3 = LinearSegmentedColormap.from_list("MarginCmap", ["#D23B33", "#FFFFFF", "#2E5B88"])

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
gdf_shifted.plot(
    ax=ax,
    column="display_margin",
    cmap=custom_cmap3,
    vmin=-0.4,
    vmax=0.4,
    legend=True,
    legend_kwds={"label": "Margin of Victory (GOP <- 0 -> DEM)", "orientation": "horizontal",
                 "shrink": 0.6, "pad": 0.02},
    edgecolor="black",
    linewidth=0.2
)
add_inset_boxes(ax, gdf_shifted)
frame_axes(ax, gdf_shifted)

ax.set_title("2026 House Contests Margin Gradient", fontsize=14)
plt.tight_layout()
plt.savefig("images/2026-Blue-Red-Gradient-Map.png", dpi=300, bbox_inches="tight", transparent=True, pad_inches=0.1)
plt.show()

In [ ]:
# -------------------------------------------------------------
# CELL 4: 2024 -> 2026 Shift Map (two-party margin)
# -------------------------------------------------------------
# shift_2p = (D-R)/(D+R) in 2026  minus  (D-R)/(D+R) in 2024, built by 1_data_collector.py.
# Districts are grouped by shift_status:
#   ok             -> coloured by shift (blue = toward DEM, red = toward GOP, white = no change)
#   redrawn        -> state redrew its lines for 2026, so the same district number
#                     covers a different electorate: hatched, not coloured
#   not_comparable -> uncontested / same-party race / missing in either year: gray
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

SHIFT_LIMIT = 0.15   # colours saturate at a 15-point swing; larger swings show at full colour

shift_cmap = LinearSegmentedColormap.from_list("ShiftCmap", ["#D23B33", "#F7F7F7", "#2E5B88"])
status = gdf_shifted["shift_status"]

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
gdf_shifted[status == "ok"].plot(
    ax=ax,
    column="shift_2p",
    cmap=shift_cmap,
    vmin=-SHIFT_LIMIT,
    vmax=SHIFT_LIMIT,
    legend=True,
    legend_kwds={"label": "Shift in two-party margin, 2024 -> 2026",
                 "orientation": "horizontal", "shrink": 0.6, "pad": 0.02, "extend": "both",
                 "format": FuncFormatter(lambda v, _: "0" if abs(v) < 1e-9
                                         else f"{'D' if v > 0 else 'R'}+{abs(v) * 100:.0f}")},
    edgecolor="black",
    linewidth=0.2
)
gdf_shifted[status == "redrawn"].plot(
    ax=ax, facecolor="#EDEDED", edgecolor="#8C8C8C", hatch="//////", linewidth=0.2
)
gdf_shifted[status == "not_comparable"].plot(
    ax=ax, facecolor="#BDBDBD", edgecolor="black", linewidth=0.2
)
add_inset_boxes(ax, gdf_shifted)
frame_axes(ax, gdf_shifted)

ax.legend(
    handles=[
        Patch(facecolor="#EDEDED", edgecolor="#8C8C8C", hatch="//////", label="Lines redrawn for 2026"),
        Patch(facecolor="#BDBDBD", edgecolor="black", label="Uncontested / not comparable"),
    ],
    loc="lower right", fontsize=9, frameon=False
)
ax.set_title("2024 -> 2026 House Shift (two-party margin, same district lines only)", fontsize=14)
plt.tight_layout()
plt.savefig("images/2026-Shift-From-2024-Map.png", dpi=300, bbox_inches="tight", transparent=True, pad_inches=0.1)
plt.show()

# Quick numeric summary of comparable districts
ok = gdf_shifted[status == "ok"]
print(f"Comparable districts: {len(ok)}  |  redrawn: {(status == 'redrawn').sum()}  |  "
      f"not comparable: {(status == 'not_comparable').sum()}")
print(f"Median shift: {ok['shift_2p'].median() * 100:+.1f} pts (positive = toward DEM)")
print(ok.nlargest(5, "shift_2p")[["race_title", "shift_2p"]].to_string(index=False))
print(ok.nsmallest(5, "shift_2p")[["race_title", "shift_2p"]].to_string(index=False))

In [ ]:
# -------------------------------------------------------------
# CELL 5: Blue-Red 3-D Map
# -------------------------------------------------------------
# Pass original UN-SHIFTED gdf so features map correctly to real geographic coordinates
# Assign RGBA color arrays for pydeck
def get_color(party):
    if party == "DEM":
        return [33, 102, 172, 220]
    elif party == "REP":
        return [178, 24, 43, 220]
    return [180, 180, 180, 220]

# Assign height based on total votes or margin depth
gdf["elevation"] = gdf["total_votes"].apply(lambda v: max(float(v), 10000.0) * 0.5)

gdf["fill_color"] = gdf["display_party"].apply(get_color)

# Convert GeoDataFrame to PyDeck compatible GeoJSON dictionary
geojson_data = json.loads(gdf.to_json())

# Define 3D Polygon/GeoJson Layer with explicit extrusion parameters
geojson_layer = pdk.Layer(
    "GeoJsonLayer",
    geojson_data,
    opacity=0.8,
    stroked=True,
    filled=True,
    extruded=True,
    wireframe=True,
    get_elevation="properties.elevation",
    elevation_scale=1.5,
    elevation_range=[0, 500000],
    get_fill_color="properties.fill_color",
    get_line_color=[255, 255, 255, 100],
    line_width_min_pixels=1,
    pickable=True
)

# Set initial camera view centered on US
view_state = pdk.ViewState(
    latitude=37.8,
    longitude=-96.0,
    zoom=3.5,
    pitch=50,
    bearing=-15
)

r = pdk.Deck(
    layers=[geojson_layer],
    initial_view_state=view_state,
    tooltip={
        "html": "<b>District:</b> {race_title}<br/>"
                "<b>Winner:</b> {winner_party}<br/>"
                "<b>Total Votes:</b> {total_votes}<br/>"
                "<b>Status:</b> {editorial_status}",
        "style": {"backgroundColor": "#1E1E1E", "color": "#FFFFFF"}
    }
)
r.to_html("images/2026-3D-turnout-map.html")